# 卷积神经网络

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
# 图表会像一张静态图片一样，直接出现在你运行代码的单元格
%matplotlib inline

In [2]:
input_size = 28
num_classes = 10
num_epochs = 3
batch_size = 64

from pathlib import Path
Path('data').mkdir(exist_ok=True)

train_dataset = datasets.MNIST(root='data', train=True,
                        transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='data', train=False,
                       transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=True)

In [4]:
train_dataset.data.shape

torch.Size([60000, 28, 28])

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU()  # 64*7*7
        )
        self.out = nn.Linear(64 * 7 * 7, num_classes)
    def forward(self, x) -> torch.Tensor:
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1) # 展平除了batch_size的其他所有维度 (batch_size, 32*7*7)
        out = self.out(x)
        return out
CNN()

CNN(
  (conv1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
  )
  (out): Linear(in_features=3136, out_features=10, bias=True)
)

In [5]:
def accuracy(predictions, labels):
    # 最大值的索引，索引就是分类结果
    pred = torch.max(predictions.data, 1)[1]
    rights = pred.eq(labels.data.view_as(pred)).sum()
    return rights, len(labels)

In [ ]:
net = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)
net.train()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net.to(device)
for epoch in range(num_epochs):
    train_rights, train_total = 0, 0
    loss = torch.tensor(0.0)
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        rights, total = accuracy(outputs, labels)
        train_rights += rights
        train_total += total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_rights/train_total:.4f}')
torch.cuda.empty_cache()

Epoch [1/3], Loss: 0.0112, Train Accuracy: 0.9488
Epoch [2/3], Loss: 0.0039, Train Accuracy: 0.9870
Epoch [3/3], Loss: 0.1122, Train Accuracy: 0.9910
